# Treatment adherence trial 1

### 1. We just consider ATP = N05A... (Antipsychotics)
- N05AD1 / N05AD01 (Haloperidol)
- N05AH02 (Clozapina)
- N05AH03 (Olanzapina)
- N05AH04 (Quetiapina)
- N05AH06 (Clotiapina)
- N05AL01 (Sulpirida)
- N05AL05 (Amisulpirida)
- N05AX08 (Risperidona)
- N05AX12 (Aripiprazol)
- N05AX13 (Paliperidona)
- N05AE05 (Lurasidona)  

In [1]:
import pandas as pd
import numpy as np
import re

# 1. Load data
df_dispensats = pd.read_csv("C:\\Users\\LRAJAG\\Desktop\\Data sintetica\\dades_sint_padris\\farmacs_dispensats.csv", sep=',', encoding='latin1')
df_mhda_tot = pd.read_csv("C:\\Users\\LRAJAG\\Desktop\\Data sintetica\\dades_sint_padris\\farmacs_MHDA.csv", sep=',', encoding='latin1')

# 2. Estandarización de las fuentes

# Farmacia comunitaria
df_disp = df_dispensats[['idCas', 'Any_Mes', 'ID_HIS_Subgrup_7_ATC', 'DES_HIS_Via_Administracio_Producte', 'ID_HIS_Forma_Farmaceutica', 'HIS_Quantitat_PA_Espec']].copy()
df_disp.columns = ['idCas', 'Mes', 'ATC', 'Via', 'Forma_Codi', 'Dosi_MG']
df_disp['Origen'] = 'Carrer'

# Farmacia hospitalaria (MHDA)
# Nota: Los antipsicóticos en MHDA suelen ser mayoritariamente LAIs (inyectables de larga duración)
df_mhda = df_mhda_tot[['idCas', 'Any_Mes','ID_HIS_Subgrup_7_ATC','DES_HIS_Forma_Farmaceutica', 'ID_HIS_Forma_Farmaceutica', 'DES_HIS_Producte']].copy()
df_mhda.columns = ['idCas', 'Mes', 'ATC', 'Via', 'Forma_Codi', 'Dosi_MG']
# Passar dosi a int (extreure només el número)
def extreure_dosi(text):
    numeros = re.findall(r'\d+', str(text)) # Busca qualsevol número en el text
    return int(numeros[0]) if numeros else 0
df_mhda['Dosi_MG'] = df_mhda['Dosi_MG'].apply(extreure_dosi)

# Creem la columna 'Via' basada en la 'Forma' real --> Class LAI o Oral
def mapejar_via(forma):
    forma = str(forma).upper()
    if 'IJ' in forma or 'INJECTBLES' in forma or 'XP' in forma or 'IP' in forma:
        return 'Intramuscular/Injectable'
    elif 'CO' in forma or 'CA' in forma or 'COMPRIMITS' in forma:
        return 'Oral'
    else:
        return 'Altres/Desconegut'
df_mhda['Via'] = df_mhda['Forma_Codi'].apply(mapejar_via)
df_disp['Via'] = df_disp['Forma_Codi'].apply(mapejar_via)
df_mhda['Origen'] = 'Hospital'

# Unimos ambas tablas
df_total = pd.concat([df_disp, df_mhda], ignore_index=True)

# 3. FILTRADO: Solo códigos que empiecen por N05
df_n05 = df_total[df_total['ATC'].str.startswith(('N05AD01', 'N05AD1',  # Haloperidol
    'N05AH02',            # Clozapina
    'N05AH03',            # Olanzapina
    'N05AH04',            # Quetiapina
    'N05AH06',            # Clotiapina
    'N05AL01',            # Sulpirida
    'N05AL05',            # Amisulpirida
    'N05AX08',            # Risperidona
    'N05AX12',            # Aripiprazol
    'N05AX13',            # Paliperidona
    'N05AE05'             # Lurasidona
    ))].copy()
print(df_n05.tail())

       idCas     Mes      ATC                       Via Forma_Codi  Dosi_MG  \
14646   1100  201908  N05AX13  Intramuscular/Injectable         IJ      150   
14647   1100  201909  N05AX13  Intramuscular/Injectable         IJ      150   
14648   1100  201910  N05AX13  Intramuscular/Injectable         IJ      150   
14649   1100  201911  N05AX13  Intramuscular/Injectable         IJ      150   
14650   1100  201912  N05AX13  Intramuscular/Injectable         IJ      150   

         Origen  
14646  Hospital  
14647  Hospital  
14648  Hospital  
14649  Hospital  
14650  Hospital  


Tenemos que comprobar que las formas de accion prolongada consideren otro tipo de adherencia.
- Usaremos la columna (HIS) Producte Desc de la tabla de MHDA para detectar si el producto es trimestral. Normalmente, los nombres comerciales o las descripciones técnicas incluyen "3 meses", "3m", "trimestral" o dosis específicas (como 350mg o 525mg en el caso de la paliperidona).
- En el tratamiento de la esquizofrenia, fármacos como la Paliperidona tienen versiones mensuales (ej. Xeplion) y trimestrales (ej. Trevicta). Si no ajustamos el código, un paciente con la versión trimestral parecería "no adherente" (solo 4 registros al año) cuando en realidad está perfectamente tratado.

In [ ]:
# 1. Definir Diccionario de Duración 

def obtener_duracion(producto):
    prod = str(producto).lower()
    if any(x in prod for x in ['3 meses', '3m', 'trimestral', 'trevicta', '525', '350']):
        return 3
    elif any(x in prod for x in ['6 meses', '6m', 'byannli']): # Existen ya de 6 meses
        return 6
    else:
        return 1

# --- 2. Aplicar a la Tabla MHDA ---
df_mhda['Duracion_Meses'] = df_mhda['(HIS) Producte Desc'].apply(obtener_duracion)

# --- 3. Lógica de Expansión ---
# Creamos una lista para los nuevos registros expandidos
registros_expandidos = []

for index, row in df_mhda.iterrows():
    id_cas = row['idCas']
    mes_inicial = int(row['Mes de l\'any Activitat'])
    duracion = int(row['Duracion_Meses'])
    
    # Por cada mes de duración, creamos un registro de cobertura
    for i in range(duracion):
        # Calculamos el mes futuro (esto es una simplificación, 
        # en datos reales habría que manejar el cambio de año 202312 -> 202401)
        mes_cobertura = mes_inicial + i 
        
        registros_expandidos.append({
            'idCas': id_cas,
            'Mes': mes_cobertura,
            'Tipo_Tratamiento': 'LAI',
            'Producto': row['(HIS) Producte Desc']
        })

df_lai_expandido = pd.DataFrame(registros_expandidos)

# --- 4. Unir con los datos Orales y Calcular ---
# (Asumiendo que df_oral ya está limpio como en el paso anterior)
df_final_estudio = pd.concat([df_oral_disp, df_lai_expandido], ignore_index=True)

# Eliminamos duplicados por si un mes se solapan (ej. le dan la inyección antes de que caduque la anterior)
adherencia_real = df_final_estudio.groupby(['idCas', 'Tipo_Tratamiento'])['Mes'].nunique().reset_index()
adherencia_real['Ratio'] = adherencia_real['Mes'] / 12

In [9]:
# --- PASO 3: Cálculo de la Adherencia (Persistencia Mensual) ---

# Definimos el periodo de estudio (ejemplo: 12 meses)
mesos_estudi = 12

# Agrupamos por paciente y tipo de tratamiento para contar meses únicos con dispensación
adherencia = df_total.groupby(['idCas', 'Tipo_Tratamiento'])['Mes'].nunique().reset_index()

# Calculamos el ratio (Meses cubiertos / Meses totales)
adherencia['Ratio_Adherencia'] = adherencia['Mes'] / mesos_estudi

# Definimos un umbral de adherencia (habitualmente 0.80 o 80%)
adherencia['Es_Adherente'] = adherencia['Ratio_Adherencia'] >= 0.80

# --- RESULTADOS ---
print("Resumen de Adherencia por Paciente:")
print(adherencia.sort_values(by='idCas'))

# Opcional: Guardar resultados
# adherencia.to_csv('resultados_adherencia_esquizofrenia.csv', index=False)

NameError: name 'df_total' is not defined